# Lab 04: Launch-Day Assimilation, Fast Readers, Turbulence & Joint Monte Carlo

Welcome to **Lab 04** of the **Tempest-ARKNET** Interactive Documentation Suite.

This lab walks through the **launch-day and 6-DOF flight-simulator stack**: decoding real airport METAR reports (including the meteorological **FROM-vs-TO** wind convention), computing observation assimilation weights (`tau_h = 6.3 h` and the `5.0 km` own-station radius), benchmarking Tempest's sub-microsecond 10 m `float32` `Atmosphere` lookup tables, comparing Dryden turbulence (`exact_ar=False` default vs. `exact_ar=True`), and running a joint **Vehicle + Atmosphere Monte Carlo & Importance Sampling** experiment with `pad_air_density` variance attribution!

> **Python Compatibility Note:** Full `tempest_weather_v2.Launch` requires Python `>= 3.12` (due to PEP 701 nested f-strings in `tempest_weather_v2/launch.py:684`). This notebook imports directly from `tempest_weather_v2.metar`, `obs`, `readers`, `montecarlo`, `weather_calc`, and `tempest_weather.flight.dryden`, which all run cleanly on Google Colab's Python 3.10/3.11 CPU runtime!

In [ ]:
# @title 0. Environment Setup (Run Once)
MOUNT_DRIVE = False # @param {type:"boolean"}
REPO_URL = "https://github.com/senai-sorsa/Tempest-ARKNET.git" # @param {type:"string"}

import importlib.util, os, sys, subprocess, pathlib

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"Drive mount skipped (not running in Colab): {exc}")

repo_dir = pathlib.Path("/content/Tempest-ARKNET")
if not repo_dir.exists():
    # Allow running inside a local checkout if already inside the repo
    cwd = pathlib.Path.cwd()
    if (cwd / "tempest").is_dir() and (cwd / "tempest_weather_v2").is_dir():
        repo_dir = cwd
    else:
        print(f"Cloning {REPO_URL} into {repo_dir} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

# Install lightweight dependencies only if missing (all are pre-installed on Google Colab)
missing_pkgs = [p for p in ("numpy", "scipy", "matplotlib") if importlib.util.find_spec(p) is None]
if missing_pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs], check=False)

print(f"Python: {sys.version.split()[0]} | Repo ready at: {repo_dir}")
if sys.version_info < (3, 12):
    print("Note: Full tempest_weather_v2.launch requires Python >= 3.12 (PEP 701 f-strings at launch.py:684).")
    print("      This lab imports Python 3.10/3.11-compatible submodules directly and runs cleanly on Colab CPU.")

## 1. Aviation METAR Parsing (`metar.py`) & Exact Weather Calculators (`weather_calc.py`)

### Conceptual Intuition (Before the Math)
On launch morning, the nearest airport broadcasts a cryptic one-line **METAR** string like `KTCS 241553Z 22014G21KT 10SM CLR 28/09 A3008 RMK AO2`. To a pilot, that's routine; to a 6-DOF rocket simulator, it must be converted into SI physics quantities: wind vector in m/s, temperature in Kelvin, true **station pressure** (inverted from the NWS altimeter setting `A3008` using the pad's exact elevation!), **density altitude**, and local **magnetic declination** (IGRF-13) so compass headings align with true north.

### Intuition Bridge: Meteorological **FROM** vs. Physics **TO** Wind Convention
One of the most notorious 180° bugs in rocket flight simulation is mixing up **meteorological** and **physics** wind directions:
- **Meteorological / METAR convention (`wind_from_deg`):** Reports the compass bearing the wind is blowing **FROM**—imagine standing at the pad looking upwind into the breeze. So `22014KT` means wind blowing **FROM 220° (southwest)** at 14 knots.
- **6-DOF Physics / Velocity vector (`u` east, `v` north):** Points in the direction the air parcel is moving **TO**—so a `220°` wind pushes air **TO 040° (northeast)**, giving **positive** $u$ (eastward) and **positive** $v$ (northward):
$$u_{\text{east}} = -V \sin(\theta_{\text{from}}), \qquad v_{\text{north}} = -V \cos(\theta_{\text{from}})$$
If you forget the minus sign (confusing FROM with TO), your rocket weathercocks into the *opposite* quadrant and your predicted landing zone flips 180° across the range!

> **Predict → Run → Observe**
> - **Predict:** For `KTCS ... 22014G21KT` (wind FROM 220° = southwest), what signs (`+` or `-`) should the physics velocity components $(u_{\text{east}}, v_{\text{north}})$ have? And at `PAD_ELEV_M = 1401 m` on a warm 28 °C morning, will **density altitude** be higher or lower than the physical pad elevation?
> - **Run:** Execute Cell 1 below, then switch `METAR_STRING` to `KDEN ... 34018G26KT` (cold 12 °C, wind from NNW) and re-run.
> - **Observe:** Verify the 180° flip between `wind_from_deg` (220° SW) and the physics TO vector (`+u, +v` toward 040° NE), and see how hot air pushes density altitude far above physical pad elevation.

In [ ]:
# @title 1. Parse METAR & Compute Station Pressure, Wind Vector (FROM→TO) & Density Altitude
METAR_STRING = "KTCS 241553Z 22014G21KT 10SM CLR 28/09 A3008 RMK AO2" # @param ["KTCS 241553Z 22014G21KT 10SM CLR 28/09 A3008 RMK AO2", "KMAF 161453Z 18008KT 10SM FEW060 31/14 A2996 RMK AO2", "KDEN 101253Z 34018G26KT 10SM SCT080 12/M02 A3018 RMK AO2"]
PAD_ELEV_M = 1401 # @param {type:"slider", min:0, max:2500, step:50}

import time
import numpy as np
import matplotlib.pyplot as plt

from tempest_weather_v2 import metar, obs, readers, montecarlo, weather_calc
from tempest_weather.flight import dryden
from tempest.profiles.to_heights import HEIGHTS
from tempest.profiles.thermo import column as thermo_column

m = metar.parse(METAR_STRING)
p_stn_hpa = weather_calc.station_pressure_hpa(m.altimeter_hpa, PAD_ELEV_M)
alt_check = weather_calc.altimeter_setting_hpa(p_stn_hpa, PAD_ELEV_M)
T_k = m.temp_c + 273.15
rho_pad = (p_stn_hpa * 100.0) / (287.05 * T_k)
da_m = weather_calc.density_altitude_m(rho_pad)
decl = weather_calc.declination_deg(32.99, -106.97, year=2026.5)

# Convert meteorological FROM bearing -> physics TO velocity vector (u east, v north)
from_rad = np.deg2rad(m.wind_from_deg)
to_deg = (m.wind_from_deg + 180.0) % 360.0
u_east = -m.wind_speed_ms * np.sin(from_rad)
v_north = -m.wind_speed_ms * np.cos(from_rad)

print(f"Station: {m.station} | METAR Wind (FROM): {m.wind_speed_ms:.2f} m/s from {m.wind_from_deg:.0f}° ({weather_calc.cardinal(m.wind_from_deg)}) | Gust: {m.gust_ms} m/s")
print(f"Physics Vector (TO): blowing toward {to_deg:.0f}° ({weather_calc.cardinal(to_deg)}) -> u_east = {u_east:+.2f} m/s, v_north = {v_north:+.2f} m/s")
print(f"Temp / Dewpoint: {m.temp_c:.1f} °C / {m.dew_c:.1f} °C | Altimeter: {m.altimeter_hpa:.2f} hPa (round-trip: {alt_check:.2f} hPa)")
print(f"Derived Station Pressure at {PAD_ELEV_M} m MSL: {p_stn_hpa:.2f} hPa")
print(f"Pad Air Density: {rho_pad:.4f} kg/m³ -> Density Altitude: {da_m:.0f} m ({da_m*3.28084:.0f} ft, +{da_m - PAD_ELEV_M:.0f} m above pad)")
print(f"IGRF-13 Magnetic Declination (Spaceport America, 2026.5): {decl:+.2f}°")
print(f"observation_kwargs(): {m.observation_kwargs()}")

## 2. Launch-Day Observation Assimilation Gain (`obs.py`)

### Conceptual Intuition (Before the Math)
Suppose an airport 35 km away reported calm wind 4 hours ago. How much should you trust that report to override your ensemble forecast at the launch pad?

Think of `tempest_weather_v2/obs.py` as a **two-regime trust-decay dimmer switch** anchored by two empirically fitted constants:
1. **Own-Station Radius (`OWN_STATION_KM = 5.0 km`) & Exponential Lead-Time Decay (`TAU_H = 6.3 h`):**
   - When a sensor sits **inside 5.0 km** of the pad (e.g., a launch-pad anemometer), it measures the *same local airmass and microclimate* as the rail. At $\Delta t = 0$, trust starts at $r = 1.0$ (exact pin) and decays purely with observation age via **`persistence_gain(age_hours)`**:
     $$r_{\text{own}}(\Delta t) = \max\!\left(e^{-\Delta t / 6.3\text{ h}},\; 0.02\right)$$
   - That **`6.3 h` e-folding time** (fitted to the GW-1 station benchmark) means a 1-hour-old pad reading still carries **~85% trust** ($e^{-1/6.3} = 0.85$), a 3-hour-old reading drops to **~62%**, and after 6.3 hours trust falls to $1/e \approx 37\%$ as synoptic weather evolves.
2. **Cross-Station Regime (`d >= 5.0 km`, Neighboring ASOS Airports):**
   - The moment a station is **$\ge 5.0\text{ km}$ away**, local terrain, thermal plumes, and drainage flows decouple it from the pad. Even a **0-minute-old** neighbor report cannot reach $r = 1.0$: `representativeness_gain` caps it at **0.66 by day / 0.52 by night** (fitted on 2015–2022 southern NM / west Texas ASOS pairs), penalizes elevation mismatch $|\Delta z|$ (scales `1317 m` day / `919 m` night), and multiplies by a slower cross-station age factor (`TAU_X = 27 h` day / `12 h` night beyond 6 h).
3. **Noise Translation:** Finally, `observation_sigma` converts gain $r \in [0.02, 1]$ into an equivalent sensor uncertainty $\sigma_{\text{obs}} = \sigma_{\text{prior}} \sqrt{1/r - 1}$ for soft Bayesian conditioning ($r = 0.5 \Rightarrow \sigma_{\text{obs}} = \sigma_{\text{prior}}$, moving the posterior median halfway).

> **Predict → Run → Observe**
> - **Predict:** On the right-hand plot (gain vs. station distance $d$), what will happen right at **$d = 5.0\text{ km}$**? Will the curve fall smoothly from 1.0, or drop with a sharp regime step? And will daytime or nighttime retain higher trust at 25 km?
> - **Run:** Execute Cell 2 below, then slide `STATION_DIST_KM` to `0.0` (own pad) and compare against `10.0` km, or change `SOLAR_HOUR` from `10` (day) to `2` (night).
> - **Observe:** Spot the crisp transition at `OWN_STATION_KM = 5.0 km`, confirm the `TAU_H = 6.3 h` exponential curve on the left, and see how nighttime cold-air pooling shrinks cross-station trust on the right.

### Interactive Controls
Move the sliders for observation age, station distance, elevation difference, and local solar hour to see how Tempest weights your launch-day measurement.

In [ ]:
# @title 2. Visualize Observation Assimilation Gain, tau_h = 6.3 h & 5.0 km Own-Station Boundary
OBS_AGE_H = 1.5 # @param {type:"slider", min:0.0, max:12.0, step:0.5}
STATION_DIST_KM = 25.0 # @param {type:"slider", min:0.0, max:100.0, step:5.0}
ELEV_DIFF_M = 200.0 # @param {type:"slider", min:0.0, max:1200.0, step:50.0}
SOLAR_HOUR = 10.0 # @param {type:"slider", min:0.0, max:23.0, step:1.0}

nw = obs.night_weight(SOLAR_HOUR)
g_sel = obs.effective_gain(age_hours=OBS_AGE_H, distance_km=STATION_DIST_KM, dz_m=ELEV_DIFF_M, night=nw)
prior_sd = 3.0  # m/s prior wind uncertainty
sig_obs = obs.observation_sigma(prior_sd, g_sel)
post_sd = prior_sd * np.sqrt(1.0 - g_sel)

print(f"obs.py constants | TAU_H = {obs.TAU_H} h (own-station e-folding) | OWN_STATION_KM = {obs.OWN_STATION_KM} km")
print(f"Own-station persistence check | Δt=1 h -> {obs.persistence_gain(1.0):.3f} | Δt=3 h -> {obs.persistence_gain(3.0):.3f} | Δt=6.3 h -> {obs.persistence_gain(obs.TAU_H):.3f} (1/e)")
print(f"Solar hour {SOLAR_HOUR:.0f}h -> night_weight = {nw:.2f} | regime = {'OWN-PAD (<5 km)' if STATION_DIST_KM < obs.OWN_STATION_KM else 'CROSS-STATION (>=5 km)'}")
print(f"Effective gain r = {g_sel:.3f} | Equivalent σ_obs = {sig_obs:.2f} m/s | Posterior σ = {post_sd:.2f} m/s (from {prior_sd:.2f} m/s)")

ages = np.linspace(0, 12, 120)
dists = np.concatenate([np.linspace(0.0, 4.99, 25), np.linspace(5.0, 100.0, 120)])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
axes[0].plot(ages, [obs.effective_gain(a, 0.0, 0.0, nw) for a in ages], lw=2.2, color="#16a34a", label=f"Own pad (d < {obs.OWN_STATION_KM:.0f} km, τ={obs.TAU_H} h)")
axes[0].plot(ages, [obs.effective_gain(a, STATION_DIST_KM, ELEV_DIFF_M, nw) for a in ages], lw=2.2, color="#2563eb", label=f"d={STATION_DIST_KM:.0f} km, Δz={ELEV_DIFF_M:.0f} m")
axes[0].axvline(obs.TAU_H, ls=":", color="#15803d", lw=1.4, label=f"τ_h = {obs.TAU_H} h (1/e ≈ 0.37)")
axes[0].scatter([OBS_AGE_H], [g_sel], color="#dc2626", s=70, zorder=5, label="Selected point")
axes[0].set(title="Assimilation Gain vs. Observation Age", xlabel="Observation Age Δt (hours)", ylabel="Effective Gain r (0..1)")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(dists, [obs.effective_gain(OBS_AGE_H, d, ELEV_DIFF_M, 0.0) for d in dists], lw=2.2, color="#ea580c", label="Daytime (night=0)")
axes[1].plot(dists, [obs.effective_gain(OBS_AGE_H, d, ELEV_DIFF_M, 1.0) for d in dists], lw=2.2, color="#4f46e5", label="Nighttime (night=1)")
axes[1].axvline(obs.OWN_STATION_KM, ls="--", color="#0f172a", lw=1.4, label=f"Own-station radius ({obs.OWN_STATION_KM:.1f} km)")
axes[1].scatter([STATION_DIST_KM], [g_sel], color="#dc2626", s=70, zorder=5)
axes[1].set(title="Spatial Representativeness Decay (Day vs. Night)", xlabel="Station Distance d (km)", ylabel="Effective Gain r (0..1)")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Fast 10 m Float32 Readers (`readers.Atmosphere`) & Dryden / von Kármán Turbulence (`dryden.py`)

### Conceptual Intuition (Before the Math)
A 6-DOF rocket simulator calls `wind(altitude)` and `air(altitude)` **tens of thousands of times per flight**. Re-interpolating irregular height arrays every millisecond would be painfully slow.

`tempest_weather_v2/readers.py::Atmosphere` solves this like a **pre-indexed stopwatch**: at construction, it expands the 44-level profile once onto a uniform **10 m `float32` grid** (plus an 86 km US Standard Atmosphere tail). Every scalar lookup is then a single array index + linear blend taking **~1 microsecond**!

On top of the smooth mean wind, `tempest_weather/flight/dryden.py` adds **fine-scale turbulence** driven by two physical sources:
- **Boundary-Layer Turbulence ($\sigma_{\text{BL}}$):** Mechanical eddies near the ground decaying toward the top of the mixing layer $h_{\text{PBL}}$.
- **Clear-Air Turbulence ($\sigma_{\text{CAT}}$):** Triggered aloft wherever wind shear overcomes thermal stability (Gradient Richardson Number $\mathrm{Ri} < 0.25$).
- Plus an optional **1-cos discrete gust** (`one_minus_cos`) for structural max-Q load testing!

### Intuition Bridge: Dryden `exact_ar=False` (Default) vs. `exact_ar=True`
How does `dryden.synthesize` step the 2nd-order transverse Dryden filter ($x_1'' + 2 x_1'/\tau + x_1/\tau^2 = n/\tau^2$) upward along the rocket's vertical path?
- **`exact_ar=False` (default, backward-compatible Euler sub-stepping):** Runs an explicit burn-in loop (`min(60*tau0, 4000) / h` steps) to reach stationarity, then takes `nsub=8` tiny Euler sub-steps ($h = dz/8$) inside every altitude step `dz`. It reproduces legacy outputs bit-for-bit, but spends extra compute on burn-in and inner sub-steps—and can drift slightly in variance if `dz` is coarse.
- **`exact_ar=True` (opt-in exact discrete AR(2) transition, `_ar2_step`):** Draws the initial state directly from the analytical stationary covariance $P = \mathrm{diag}(1/(4\tau_0),\, 1/(4\tau_0^3))$ (**zero burn-in!**) and advances in a single closed-form matrix-exponential transition $\Phi(dz/\tau)$ with exact Cholesky process noise $Q = P - \Phi P \Phi^\top$. It stays mathematically stationary at *any* step size `dz` and runs noticeably faster!

> **Predict → Run → Observe**
> - **Predict:** Between `exact_ar=False` (default Euler with burn-in + `nsub=8` sub-steps) and `exact_ar=True` (single-step analytical AR(2)), which one will run **faster** at `dz = 5.0 m`? Should their turbulence RMS standard deviations still match closely?
> - **Run:** Execute Cell 3 below to benchmark both `Atmosphere.wind()` and `dryden.synthesize(..., exact_ar=False)` vs. `exact_ar=True`, then switch `SPECTRUM` to `"karman"` or change `GUST_QUANTILE`.
> - **Observe:** Check the printed speedup of `exact_ar=True` over `exact_ar=False` alongside their matching turbulence RMS, and spot the smooth `1-cos` discrete gust pulse in the right panel.

### Interactive Controls
Switch between `"dryden"` and `"karman"`, toggle `EXACT_AR` for the plotted profile, and tune `GUST_QUANTILE` to inspect the synthesized fine-scale wind profile.

In [ ]:
# @title 3. Benchmark 10 m Atmosphere Reader & Compare Dryden exact_ar=False vs. exact_ar=True
SPECTRUM = "dryden" # @param ["dryden", "karman"]
EXACT_AR = True # @param {type:"boolean"}
GUST_QUANTILE = 0.5 # @param {type:"slider", min:0.0, max:1.0, step:0.1}

# Build a synthetic ensemble of 120 coupled atmospheres on the 44-height grid
rng = np.random.default_rng(2026)
N_ENS = 120
z_agl = HEIGHTS.astype(np.float64)
elev_m = float(PAD_ELEV_M)

u_ens = np.zeros((N_ENS, len(z_agl)), dtype=np.float32)
v_ens = np.zeros((N_ENS, len(z_agl)), dtype=np.float32)
T_ens = np.zeros((N_ENS, len(z_agl)), dtype=np.float32)
RH_ens = np.zeros((N_ENS, len(z_agl)), dtype=np.float32)
p_ens = np.zeros((N_ENS, len(z_agl)), dtype=np.float32)

for k in range(N_ENS):
    u0 = rng.normal(5.0, 2.5)
    v0 = rng.normal(1.0, 2.0)
    jet = rng.normal(20.0, 5.0) * np.exp(-0.5 * ((z_agl - 11000.0) / 3000.0) ** 2)
    u_ens[k] = u0 + jet + 1.5 * np.log1p(z_agl / 150.0)
    v_ens[k] = v0 + 5.0 * np.sin(z_agl / 4500.0)
    t_sfc = rng.normal(300.0, 4.0)
    T_ens[k] = np.where(z_agl <= 12000.0, t_sfc - 0.0066 * z_agl, t_sfc - 0.0066 * 12000.0 + 0.001 * (z_agl - 12000.0))
    RH_ens[k] = np.clip(rng.normal(30.0, 10.0) * np.exp(-z_agl / 6000.0) + 8.0, 2.0, 95.0)
    ps = 101325.0 * (1.0 - 2.25577e-5 * elev_m) ** 5.25588 + rng.normal(0.0, 350.0)
    p_ens[k] = thermo_column(z_agl, T_ens[k], RH_ens[k], ps, t_sfc=t_sfc, rh_sfc=RH_ens[k, 0], z_sfc=elev_m)["p"]

samples = dict(height_agl=z_agl, elevation_m=elev_m, u=u_ens, v=v_ens, T=T_ens, RH=RH_ens, p=p_ens)

# Construct fast 10 m float32 Atmosphere reader and benchmark 10,000 scalar calls
atm0 = readers.Atmosphere(samples, 0, frame="une", units="si", reference="asl")
t0 = time.perf_counter()
for h in np.linspace(elev_m, elev_m + 10000.0, 10000):
    _ = atm0.wind(float(h))
dt_us = (time.perf_counter() - t0) * 1e6 / 10000.0
print(f"Atmosphere.wind() scalar lookup benchmark: {dt_us:.2f} µs/call (10,000 calls)")

# Head-to-head benchmark: default exact_ar=False (Euler sub-steps + burn-in) vs. exact_ar=True (exact AR(2))
t_e0 = time.perf_counter()
turb_euler = dryden.synthesize(z_agl, u_ens[:16], v_ens[:16], T_ens[:16], dz=5.0, seed=42, z_top=6000.0, spectrum="dryden", exact_ar=False)
ms_euler = (time.perf_counter() - t_e0) * 1e3

t_a0 = time.perf_counter()
turb_exact = dryden.synthesize(z_agl, u_ens[:16], v_ens[:16], T_ens[:16], dz=5.0, seed=42, z_top=6000.0, spectrum="dryden", exact_ar=True)
ms_exact = (time.perf_counter() - t_a0) * 1e3

print(f"Dryden 16-profile benchmark (0-6 km, dz=5 m):")
print(f"  exact_ar=False (default Euler nsub=8): {ms_euler:6.1f} ms | u_turb RMS = {np.std(turb_euler['u_turb']):.3f} m/s")
print(f"  exact_ar=True  (exact AR(2), 0 burn-in): {ms_exact:6.1f} ms | u_turb RMS = {np.std(turb_exact['u_turb']):.3f} m/s ({ms_euler/max(ms_exact,1e-6):.1f}x faster)")

# Synthesize selected profile + optional 1-cos discrete gust for plotting
# gust=(ua, ul, ud, um) are unit-cube coordinates for altitude, half-length, direction, and amplitude
turb = dryden.synthesize(
    z_agl, u_ens[0], v_ens[0], T_ens[0],
    dz=5.0, seed=42, z_top=6000.0, spectrum=SPECTRUM, exact_ar=bool(EXACT_AR),
    gust=(0.30, 0.45, 0.15, float(GUST_QUANTILE)) if GUST_QUANTILE > 0 else None,
)
z_fine = turb["z"]
u_mean_fine = np.interp(z_fine, z_agl, u_ens[0])
u_tot_fine = np.asarray(turb["u_total"]).reshape(-1)
du_fine = np.asarray(turb["u_turb"] + turb["gust_u"]).reshape(-1)
dv_fine = np.asarray(turb["v_turb"] + turb["gust_v"]).reshape(-1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
axes[0].plot(u_mean_fine, z_fine / 1000.0, lw=2.2, color="#1e293b", label="Mean u(z)")
axes[0].plot(u_tot_fine, z_fine / 1000.0, lw=1.0, alpha=0.85, color="#2563eb", label=f"Mean + {SPECTRUM} (exact_ar={EXACT_AR}) + gust")
axes[0].set(title=f"Synthesized {SPECTRUM.title()} Wind Profile (0–6 km AGL)", xlabel="Eastward Wind u (m/s)", ylabel="Altitude (km AGL)")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(du_fine, z_fine / 1000.0, lw=1.0, color="#ea580c", label="Δu (turb + gust)")
axes[1].plot(dv_fine, z_fine / 1000.0, lw=1.0, color="#0284c7", alpha=0.75, label="Δv (turb + gust)")
axes[1].axvline(0, color="k", lw=0.8)
axes[1].set(title="Turbulence + 1-cos Discrete Gust Fluctuations", xlabel="Wind Fluctuation (m/s)", ylabel="Altitude (km AGL)")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Joint Vehicle + Atmosphere Monte Carlo & Importance Sampling (`montecarlo.py`)

### Conceptual Intuition (Before the Math)
A common mistake in rocket dispersion analysis is varying *only* the weather—when in reality, flight-test validation showed **vehicle uncertainty** (drag coefficient, thrust, dry mass) is often **2–4× larger** than atmospheric uncertainty!

`tempest_weather_v2/montecarlo.py` handles both together:
- **`VehiclePriors` + `montecarlo.run`:** Pairs each sampled `Atmosphere` with a Latin Hypercube (`lhs`) or `sobol` draw of vehicle parameters, then computes **variance attribution** (`mc.contributions()` and joint rank regression `mc.explained()`).
- **Intuition Bridge — Why `pad_air_density` Matters in `mc.contributions()`:** An ensemble member's integer index (`atm_index`) has no physical meaning—correlating drift against `atm_index` produces pure noise! So `montecarlo.atmosphere_metrics(atm)` automatically extracts three physical atmospheric scalars for every member:
  1. **`atmosphere: pad_air_density`** (air density $\rho$ at `elevation_m + 1 m`, governing aerodynamic drag $q = \frac{1}{2}\rho v^2$ and apogee altitude),
  2. **`atmosphere: mean_wind_0_200m`** (near-pad wind governing rail-exit weathercocking), and
  3. **`atmosphere: mean_wind_0_3km`** (bulk tropospheric wind governing parachute descent drift).
  Separating `pad_air_density` from wind lets you immediately tell whether a dispersion spike is a **thermodynamic drag/apogee problem** or a **kinematic wind-drift problem**!
- **`montecarlo.run_importance`:** Want to estimate a rare **1-in-200 range-safety exceedance** without running 10,000 expensive 6-DOF simulations? Importance sampling tilts the atmospheric sampling weights toward high-wind days (`metric="mean_wind_0_3km"`), runs only `n = 120` simulations, and mathematically re-weights them back to an exact unbiased exceedance probability!

> **Predict → Run → Observe**
> - **Predict:** In our landing-drift model (where drift = wind speed × descent time), which of the three atmospheric keys (`mean_wind_0_3km`, `mean_wind_0_200m`, or `pad_air_density`) will dominate variance attribution? And if you raise `DRAG_SIGMA` from `0.06` to `0.15`, will `drag_scale` overtake `pad_air_density`?
> - **Run:** Execute Cell 4 below, then increase `DRAG_SIGMA` to `0.15` or switch `MC_METHOD` to `"sobol"` and re-run.
> - **Observe:** Inspect the `"atmosphere: pad_air_density"` bar alongside the wind and vehicle bars, check `mc.explained()` (`r2_vehicle` vs. `atmosphere_gain`), and compare the importance-sampling tail estimate against standard MC.

### Interactive Controls
Choose the sampling method (`lhs` or `sobol`), set the range-safety landing-drift limit, and adjust `DRAG_SIGMA` to inspect variance attribution and importance-sampling efficiency.

In [ ]:
# @title 4. Run Joint Vehicle + Atmosphere MC, pad_air_density Attribution & Importance Sampling
MC_METHOD = "lhs" # @param ["lhs", "sobol"]
DRIFT_LIMIT_M = 1600.0 # @param {type:"slider", min:800, max:2600, step:100}
DRAG_SIGMA = 0.06 # @param {type:"slider", min:0.01, max:0.15, step:0.01}

# Build list of 120 fast Atmosphere readers (SI, ASL)
atms = [readers.Atmosphere(samples, k, frame="une", units="si", reference="asl") for k in range(N_ENS)]

priors = montecarlo.VehiclePriors(
    drag_scale=("lognormal", float(DRAG_SIGMA)),
    mass_scale=("normal", 0.025),
    thrust_scale=("normal", 0.03),
)

# Fast analytic rocket landing-drift surrogate (metres downrange)
def simulate_drift(atm, drag_scale=1.0, mass_scale=1.0, thrust_scale=1.0):
    # Sample wind at 5 altitudes from 100 m to 3000 m AGL and pad air density
    alts = elev_m + np.array([100.0, 500.0, 1200.0, 2000.0, 3000.0])
    winds = atm.wind_many(alts)  # (5, 3) -> (up, east, north)
    mean_ws = float(np.mean(np.hypot(winds[:, 1], winds[:, 2])))
    _, _, rho0, _ = atm.air(elev_m + 10.0)
    # Approximate apogee & parachute drift sensitivity
    apogee_m = 3200.0 * (thrust_scale ** 1.4) / (drag_scale ** 0.55 * mass_scale ** 0.7 * (rho0 / 1.05) ** 0.35)
    descent_time_s = apogee_m / 22.0
    drift_m = mean_ws * descent_time_s * 0.95
    return float(drift_m)

mc = montecarlo.run(simulate_drift, atms, priors=priors, n=120, seed=42, method=MC_METHOD)
is_res = montecarlo.run_importance(simulate_drift, atms, threshold=float(DRIFT_LIMIT_M), priors=priors, n=120, tilt=1.8, metric="mean_wind_0_3km", seed=42, method=MC_METHOD)

p_std = float(np.mean(mc.values > DRIFT_LIMIT_M))
p_is, p_is_se = is_res.probability()
contribs = mc.contributions()
expl = mc.explained()

print(f"Joint MC ({MC_METHOD.upper()}, n=120): 90% Drift Interval = [{mc.interval(90)[0]:.0f}, {mc.interval(90)[1]:.0f}] m")
print(f"P(Drift > {DRIFT_LIMIT_M:.0f} m) | Standard MC: {100*p_std:.2f}% | Importance Sampling: {100*p_is:.2f}% ± {100*p_is_se:.2f}% (N_eff={is_res.effective_n():.1f})")
print(f"Joint Rank-R² | Vehicle only: {100*expl['r2_vehicle']:.1f}% | All (Vehicle + Atm): {100*expl['r2_all']:.1f}% | Atmosphere gain: +{100*expl['atmosphere_gain']:.1f}%")
print(f"Physical atmosphere metrics extracted: {list(mc.atm_metrics.keys())} (pad_air_density key present: {'pad_air_density' in mc.atm_metrics})")
print("Variance Attribution (first-order rank-R² contributions):")
for k, v in sorted(contribs.items(), key=lambda kv: -kv[1]):
    print(f"  {k:55s}: {100*v:5.1f}%")

# Also demonstrate vehicle-error deconvolution from flight-test residuals
demo_err_pct = np.array([4.2, -5.8, 3.1, -2.9, 6.4, -4.5])
deconv = montecarlo.calibrate_vehicle_sigma(demo_err_pct, atmosphere_sd_pct=np.full_like(demo_err_pct, 1.8))
print(f"Deconvolved vehicle σ from flight tests (atm_sd=1.8%): {deconv}")

# Plot only physical drivers (omit raw atm_index noise key for visual clarity)
phys_items = [(k, 100.0 * v) for k, v in contribs.items() if "sample index" not in k]
phys_items.sort(key=lambda kv: kv[1])
labels = [k for k, _ in phys_items]
vals = [v for _, v in phys_items]
colors = ["#2563eb" if lbl.startswith("atmosphere:") else "#059669" for lbl in labels]

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))
axes[0].hist(mc.values, bins=24, color="#2563eb", alpha=0.75, edgecolor="white")
axes[0].axvline(DRIFT_LIMIT_M, ls="--", color="#dc2626", lw=2.0, label=f"Range limit ({DRIFT_LIMIT_M:.0f} m)")
axes[0].set(title=f"Joint MC Landing Drift Distribution ({MC_METHOD.upper()})", xlabel="Landing Drift (m)", ylabel="Count")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].barh(labels, vals, color=colors, alpha=0.85)
axes[1].set(title="Variance Attribution (Blue=Atmosphere, Green=Vehicle)", xlabel="Explained Rank-R² (%)")
axes[1].grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

**How to interpret the plot:**
- **Left:** The histogram shows the full joint landing-drift distribution; the dashed red line marks your range-safety limit, where `run_importance` gives a tighter tail probability estimate than standard MC at the same simulation budget.
- **Right:** The horizontal bar chart decomposes landing-drift variance into **atmospheric drivers** (blue: `atmosphere: mean_wind_0_3km`, `atmosphere: mean_wind_0_200m`, `atmosphere: pad_air_density`) vs. **vehicle drivers** (green: `drag_scale`, `mass_scale`, `thrust_scale`)—try increasing `DRAG_SIGMA` to watch vehicle uncertainty overtake weather!